# Surrogate model evaluation

Runs the exact same offline accuracy evaluation as `evaluator.py`'s `evaluate_surrogate()` -- the same
function used by the `evaluate_surrogate` CLI command -- on every checkpoint currently trained, and
renders the results inline instead of writing JSON + PNG files to disk. Nothing here reimplements the
metrics: every number and every plot below comes from calling `evaluate_surrogate()` itself.
The report separates valid-beam regression from the discontinuous all-particles-lost cliff,
shows native and standardized errors, bootstrap intervals, classifier PR/calibration diagnostics,
checkpoint provenance, and the ensemble mean when multiple checkpoints are available.

When a shared `failure_classifier_*.pt` checkpoint is also found (trained alongside the surrogate
ensemble by `trainer.py`), its precision/recall/F1 at separating true all-particles-lost samples, plus
a "gated" score-metrics block, are reported too -- see section 2 and
[`visualize_surrogate_model.ipynb` section 6](visualize_surrogate_model.ipynb) for why that classifier
exists.

This notebook is read-only: it loads existing checkpoints/datasets and evaluates them in memory, it
does not train, fine-tune, or write anything back to `trained_models/`.

**Related notebooks:**
- [`visualize_surrogate_model.ipynb`](visualize_surrogate_model.ipynb) -- the `ModularMLP` architecture
  being evaluated here (layers, input/output contract), and the `FailureClassifier` (section 6).
- [`env/dataset/visualize_dataset.ipynb`](../../../dataset/visualize_dataset.ipynb) -- the `BeamDataset`
  format the evaluation dataset below is loaded from.

In [ ]:
from pathlib import Path
import sys
import tempfile

import numpy as np
import torch
from IPython.display import Image, Markdown, display

WORKING_DIR = Path.cwd().resolve()
if (WORKING_DIR / 'beam_optimization').is_dir():
    REPO_ROOT = WORKING_DIR
else:
    PROJECT_ROOT = WORKING_DIR
    while PROJECT_ROOT.name != 'beam_optimization' and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
    REPO_ROOT = PROJECT_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from beam_optimization.config.paths import DEFAULT_BASE_SURROGATE_DIR, DEFAULT_UPDATED_SURROGATE_DIR
from beam_optimization.env.dataset import BeamDataset
from beam_optimization.env.surrogate_env.surrogate.model.modular_mlp import ModularMLP
from beam_optimization.env.surrogate_env.surrogate.model.failure_classifier import FailureClassifier
from beam_optimization.env.surrogate_env.surrogate.model.trainer import compute_normalization_metadata
from beam_optimization.env.surrogate_env.surrogate.model.evaluator import (
    evaluate_surrogate, _default_test_dataset_path, _MeanEnsemble, _checkpoint_provenance,
)


def fmt(value):
    if value is None:
        return 'n/a'
    return f'{float(value):.6g}'


def table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines.extend('| ' + ' | '.join(str(cell) for cell in row) + ' |' for row in rows)
    return '\n'.join(lines)

## Modify here

Pin a specific surrogate/classifier directory and/or dataset test split below, or leave both at
their defaults to auto-discover the latest ones (same convention every script in this project uses).

In [ ]:
# <-- MODIFY HERE to inspect a different surrogate ensemble / classifier, or a different dataset.
MODEL_DIR = DEFAULT_BASE_SURROGATE_DIR
# e.g. MODEL_DIR = Path('beam_optimization/env/surrogate_env/surrogate/trained_models/updated')

DATASET_PATH_OVERRIDE = None
# e.g. DATASET_PATH_OVERRIDE = Path('beam_optimization/env/dataset/013/dataset_test.pt')
# None = auto (newest numbered dataset that actually has a dataset_test.pt split)

CLASSIFIER_THRESHOLD = 0.5
# Tune this value on dataset_val.pt, then keep it fixed for the final test.

## 1. Checkpoints and evaluation dataset

`evaluate_surrogate_folder()` (what the `evaluate_surrogate` CLI command calls) evaluates every
`surrogate_*.pt` file in one directory -- `MODEL_DIR` above (`trained_models/base/` by default; point
it at `trained_models/updated/` to inspect the online-fine-tuned ensemble instead). The dataset uses
`_default_test_dataset_path()`, the same helper `evaluator.py` itself uses, unless overridden by
`DATASET_PATH_OVERRIDE` above.

In [ ]:
checkpoint_paths = sorted(MODEL_DIR.glob('surrogate_*.pt')) if MODEL_DIR.exists() else []
checkpoint_source = str(MODEL_DIR) if checkpoint_paths else None

DATASET_PATH = DATASET_PATH_OVERRIDE or _default_test_dataset_path()
dataset = BeamDataset.load(DATASET_PATH)

print(f'{MODEL_DIR}: {len(checkpoint_paths)} checkpoint(s)')
print(f'evaluating {len(checkpoint_paths)} checkpoint(s) from: {checkpoint_source}')

models = []  # list of (name, model, checkpoint_path)
if checkpoint_paths:
    for path in checkpoint_paths:
        models.append((path.name, ModularMLP.load(str(path)), path))
else:
    norm_stats = compute_normalization_metadata(dataset)
    models.append(('untrained (demo)', ModularMLP(norm_stats=norm_stats), None))
    print(f'\nNo trained checkpoint found in {MODEL_DIR} yet (train one with the `train_surrogate` '
          f'command, or point MODEL_DIR above at a directory that already has one). Evaluating a '
          f'freshly-initialized ModularMLP instead so the rest of this notebook still runs end to '
          f'end -- every metric and plot below will reflect random weights, not real surrogate '
          f'accuracy, until a real checkpoint exists.')

# The shared FailureClassifier (trainer.py trains one per run, independent of ensemble size) lives
# alongside the surrogate_*.pt checkpoints in MODEL_DIR, named failure_classifier_<dataset>.pt.
# Optional: when missing, the rest of the notebook simply reports the regression metrics without the
# classifier section, exactly like evaluate_surrogate() does when classifier=None.
dataset_id = DATASET_PATH.parent.name
classifier_ckpts = sorted(MODEL_DIR.glob(f'failure_classifier_{dataset_id}.pt')) if MODEL_DIR.exists() else []
if classifier_ckpts:
    classifier = FailureClassifier.load(str(classifier_ckpts[0]))
    classifier.eval()
    print(f'\nLoaded shared failure classifier: {classifier_ckpts[0].name}')
else:
    classifier = None
    print(f'\nNo failure_classifier_*.pt found in {MODEL_DIR} -- '
          f'classifier_metrics/score_metrics_gated will be skipped below.')

print('\nProvenance:')
print(f'  test dataset: {DATASET_PATH.resolve()}')
for name, _, path in models:
    if path is not None:
        provenance = _checkpoint_provenance(path)
        print(f"  {name}: modified={provenance['modified_utc']}, "
              f"train={provenance['train_dataset_path']}, val={provenance['val_dataset_path']}")

## 2. Running `evaluate_surrogate()`

Calls the real function once per model, exactly as `evaluate_surrogate_folder()` does internally:
batched forward passes over the whole dataset, per-(stage, feature) SSE/SAE accumulated in native
physical units, then reduced to MSE/MAE/RMSE per stage, per feature, and overall, plus final-score
metrics (MAE, RMSE, bias, Pearson r, R²) via the project's real `score_tensor()`. `plots_dir` points at
a throwaway temporary directory -- the same plotting code `evaluator.py` uses to save PNGs to disk is
reused untouched, the files are just displayed inline here and discarded afterward instead of being
kept under `results/`.

When a shared `failure_classifier_*.pt` was found in section 1, it is also passed in: `evaluate_surrogate()`
then additionally reports how well it separates true all-particles-lost samples from the rest
(`classifier_metrics` -- precision/recall/F1/confusion matrix) and a second "gated" score-metrics block
computed as if the classifier's override (see `surrogate_simulator.run_surrogate_forward()`) had been
applied to every prediction (`score_metrics_gated`). This never changes the plain `score_metrics` above
it -- it is purely an additional diagnostic, see
[`visualize_surrogate_model.ipynb` section 6](visualize_surrogate_model.ipynb#6.-FailureClassifier:-gating-the-all-particles-lost-cliff)
for why the classifier exists and how it's trained.

In [ ]:
all_results = {}
with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    for name, model, checkpoint_path in models:
        all_results[name] = evaluate_surrogate(
            model, dataset, plots_dir=tmp_dir, plot_prefix=Path(name).stem,
            classifier=classifier, classifier_threshold=CLASSIFIER_THRESHOLD,
        )
        if checkpoint_path is not None:
            all_results[name]['checkpoint_provenance'] = _checkpoint_provenance(checkpoint_path)
        # Plot PNGs must be read into memory now, inside the `with` block, before the
        # temporary directory (and the files in it) are deleted on exit.
        for plot_name, plot_path in all_results[name]['plots'].items():
            all_results[name]['plots'][plot_name] = Path(plot_path).read_bytes()

    if len(models) > 1 and all(path is not None for _, _, path in models):
        ensemble = _MeanEnsemble([model for _, model, _ in models])
        all_results['ensemble_mean'] = evaluate_surrogate(
            ensemble, dataset, plots_dir=tmp_dir, plot_prefix='ensemble_mean',
            classifier=classifier, classifier_threshold=CLASSIFIER_THRESHOLD,
        )
        all_results['ensemble_mean']['ensemble_size'] = len(models)
        for plot_name, plot_path in all_results['ensemble_mean']['plots'].items():
            all_results['ensemble_mean']['plots'][plot_name] = Path(plot_path).read_bytes()

print(f'Evaluated {len(all_results)} report(s) on {len(dataset):,} samples from {DATASET_PATH.name}.')
if len(models) == 1:
    print('Only one checkpoint is available: ensemble uncertainty cannot be evaluated yet.')

## 3. Report, per model

In [ ]:
for name, metrics in all_results.items():
    groups = metrics['sample_groups']
    rl_groups = metrics['rl_sample_groups']
    summary_rows = [
        ['Samples', f"{metrics['n_samples']:,}"],
        ['Physical non-failure / all lost', f"{groups['n_valid']:,} / {groups['n_failures']:,}"],
        ['RL-valid (>= 10%) / terminal (< 10%)', f"{rl_groups['n_valid']:,} / {rl_groups['n_terminal']:,}"],
        ['RMSE all stages (native units)', fmt(metrics['rmse_all'])],
        ['NRMSE all stages (dimensionless)', fmt(metrics['nrmse_all'])],
        ['RMSE final stage (native units)', fmt(metrics['rmse_final_stage'])],
    ]
    score_rows = []
    for label, key, count in [
        ('All samples (regressor only)', 'score_metrics', metrics['n_samples']),
        ('Physical non-failure (npart > 0)', 'score_metrics_valid', groups['n_valid']),
        ('All particles lost (npart = 0)', 'score_metrics_failures', groups['n_failures']),
        ('RL-valid (npart >= 0.10)', 'score_metrics_rl_valid', rl_groups['n_valid']),
        ('RL-terminal (npart < 0.10)', 'score_metrics_rl_terminal', rl_groups['n_terminal']),
    ]:
        values = metrics[key]
        mae_ci = values.get('confidence_intervals_95', {}).get('mae', {})
        mae_ci_text = (f"[{fmt(mae_ci.get('low'))}, {fmt(mae_ci.get('high'))}]" if mae_ci else 'n/a')
        score_rows.append([
            label, f'{count:,}', fmt(values['mae']), mae_ci_text, fmt(values['rmse']),
            fmt(values['bias']), fmt(values['pearson_correlation']), fmt(values['r2']),
        ])
    feature_rows = [
        [f"`{feature}`", fmt(values['rmse_all_stages']), fmt(values['mae_all_stages']),
         fmt(values['rmse_final_stage']), fmt(values['nrmse_final_stage']),
         fmt(values['target_std_final_stage'])]
        for feature, values in metrics['feature_metrics'].items()
    ]
    stage_rows = [
        [marker, fmt(rmse), fmt(nrmse)]
        for marker, rmse, nrmse in zip(
            metrics['stage_markers'], metrics['rmse_per_stage'], metrics['nrmse_per_stage']
        )
    ]
    terminal_rows = []
    for label, key in [('npart regressor only', 'regressor_only'),
                       ('Current pipeline: npart regressor OR classifier', 'with_classifier_gate')]:
        values = metrics['rl_terminal_metrics'].get(key)
        if values is None:
            continue
        confusion = values['confusion_matrix']
        terminal_rows.append([
            label, fmt(values['precision']), fmt(values['recall']), fmt(values['specificity']),
            fmt(values['f1']), fmt(values['balanced_accuracy']),
            f"{confusion['tp']} / {confusion['fp']} / {confusion['fn']} / {confusion['tn']}",
        ])
    band_rows = [
        [row['interval'], f"{row['n_samples']:,}", fmt(row['true_mean']),
         fmt(row['predicted_mean']), fmt(row['mae']), fmt(row['bias']),
         fmt(row['regressor_terminal_rate']), fmt(row['classifier_flag_rate']),
         fmt(row['pipeline_terminal_rate'])]
        for row in metrics['npart_ratio_bands']
    ]
    sections = [
        f'### `{name}`',
        '**Summary**', table(['Metric', 'Value'], summary_rows),
        '**Score metrics split by physical regime**',
        table(['Group', 'N', 'MAE', 'MAE 95% CI', 'RMSE', 'Bias', 'Pearson r', 'R²'], score_rows),
        '> The all-sample regressor metric includes the discontinuous ERROR_SCORE cliff. '
        'For RL, use the npart >= 0.10 row: physical non-failure (npart > 0) is a different split.',
        '**RL terminal decision at `npart_ratio < 0.10`** — exactly 0.10 remains valid.',
        table(['Decision source', 'Precision', 'Recall', 'Specificity', 'F1',
               'Balanced accuracy', 'TP / FP / FN / TN'], terminal_rows),
        '**Final `npart_ratio` by operational band**',
        table(['True band', 'N', 'True mean', 'Predicted mean', 'MAE', 'Bias',
               'Regressor terminal rate', 'Classifier flag rate', 'Pipeline terminal rate'], band_rows),
        '> A terminal rate should be high in the first two rows and low in the RL-valid rows. '
        'The classifier is trained only for npart = 0; the continuous npart regressor carries the 10% decision.',
        '**Per-feature errors**',
        table(['Feature', 'RMSE all', 'MAE all', 'RMSE final', 'NRMSE final', 'Target std final'], feature_rows),
        '**Per-stage errors** — raw RMSE mixes physical units; NRMSE is the comparable column.',
        table(['Stage marker', 'RMSE native', 'NRMSE'], stage_rows),
    ]
    provenance = metrics.get('checkpoint_provenance')
    if provenance:
        sections += ['**Checkpoint provenance**', table(['Field', 'Value'], [
            ['Path', provenance['path']], ['Modified UTC', provenance['modified_utc']],
            ['Training dataset', provenance['train_dataset_path']],
            ['Validation dataset', provenance['val_dataset_path']],
            ['Best validation loss', fmt(provenance['best_val_loss'])],
        ])]
    if metrics.get('classifier_metrics'):
        cm = metrics['classifier_metrics']; diag = metrics['classifier_diagnostics']
        gated = metrics['score_metrics_gated']
        cm_rows = [
            ['Precision', fmt(cm['precision'])], ['Recall', fmt(cm['recall'])],
            ['F1', fmt(cm['f1'])], ['Accuracy', fmt(cm['accuracy'])],
            ['Average precision', fmt(diag['average_precision'])],
            ['Brier score', fmt(diag['brier_score'])],
            ['Expected calibration error', fmt(diag['expected_calibration_error'])],
            ['Confusion matrix (tp/fp/fn/tn)',
             f"{cm['confusion_matrix']['tp']:.0f} / {cm['confusion_matrix']['fp']:.0f} / "
             f"{cm['confusion_matrix']['fn']:.0f} / {cm['confusion_matrix']['tn']:.0f}"],
        ]
        gated_rows = [['MAE', fmt(gated['mae'])], ['RMSE', fmt(gated['rmse'])],
                      ['Bias', fmt(gated['bias'])], ['Pearson r', fmt(gated['pearson_correlation'])],
                      ['R²', fmt(gated['r2'])]]
        sweep_rows = [
            [fmt(row['threshold']), fmt(row['precision']), fmt(row['recall']),
             fmt(row['gated_score_mae']), fmt(row['gated_score_r2'])]
            for row in diag['threshold_diagnostics']
        ]
        sections += [
            f"**Failure classifier at the fixed threshold `{cm['threshold']}`**",
            table(['Metric', 'Value'], cm_rows),
            '**Full pipeline with classifier gate**', table(['Metric', 'Value'], gated_rows),
            '**Threshold diagnostics (descriptive only)**',
            '> ⚠️ Choose the threshold on `dataset_val.pt`; never select it from this final test table.',
            table(['Threshold', 'Precision', 'Recall', 'Gated MAE', 'Gated R²'], sweep_rows),
        ]
    display(Markdown('\n\n'.join(sections)))
    for plot_name, plot_bytes in metrics['plots'].items():
        display(Image(data=plot_bytes))

## 4. Where this fits in the pipeline

- This notebook is the interactive, always-available counterpart to running
  `python -m beam_optimization.env.surrogate_env.surrogate.model.evaluator --model-dir <dir> --dataset <path>`
  directly -- same `evaluate_surrogate()` call, same numbers, just rendered here instead of written to
  `results/benchmark/surrogate_eval.json` and standalone PNGs.
- See [`visualize_surrogate_model.ipynb`](visualize_surrogate_model.ipynb) for what the model being
  evaluated actually looks like (layers, parameters, forward pass), and `trainer.py` for how a real
  checkpoint gets produced in the first place (`train_surrogate` command).
- If section 1 above evaluated an `'untrained (demo)'` model, none of the numbers or plots in this
  notebook reflect real surrogate accuracy -- re-run the notebook once a checkpoint exists under
  `trained_models/base/` or `trained_models/updated/`.